# Clonogenic Photo Pipeline

Ноутбук для подсчета колоний на фотографиях 6-well планшетов.

Что делает ноутбук:
1. Загружает фотографии.
2. Делит каждое фото на 6 лунок.
3. Считает окрашенные колонии.
4. Создает картинки с контурами и номерами колоний.
5. При наличии контроля строит CDI-таблицу и heatmap.

Выполняйте ячейки сверху вниз. Все важные настройки находятся в следующей ячейке.

## Как называть файлы

Используйте латиницу, цифры, `_` или `-`. Десятичные дроби можно писать через запятую или точку.

**Одна концентрация на весь планшет, 6 повторностей:**

`<sample>_<dose>gy_<concentration>.jpg`

Примеры: `HF_0gy_0.jpg`, `HF_2gy_0,075.jpg`, `A549_HF_4gy_0.15.jpg`

**Две концентрации на планшете, по 3 повторности:**

`<sample>_<dose>gy_<right concentration>_<left concentration>.jpg`

Примеры: `HF_0gy_0,3_0,15.jpg`, `HF_2gy_0_0,075.jpg`

Лунки считаются так: `W1/W3/W5` слева, `W2/W4/W6` справа. Если указана одна концентрация, все 6 лунок считаются повторностями одного условия.

In [ ]:
# =========================
# ГЛАВНЫЕ НАСТРОЙКИ
# =========================

# Ссылка на GitHub-репозиторий с этим пайплайном.
REPO_URL = "https://github.com/KIT-MAKSGOR-Vaillka/Clonogenic.git"

# Название клеточной линии. Оно попадет в CDI CSV и заголовок heatmap-панели.
CELL_LINE_NAME = "4T1"

# Название образца/материала для заголовка heatmap.
MATERIAL_NAME = "HF"

# Папка, куда будут распакованы фотографии.
PHOTO_DIR = "data/photos"

# Папка с результатами.
OUTPUT_DIR = "results/photo_run"

# Раскладка повторностей:
# auto = если в имени 1 концентрация, все 6 лунок одно условие; если 2 концентрации, левая/правая колонки.
# split-columns = всегда считать W1/W3/W5 и W2/W4/W6 как две группы.
# all-wells = все 6 лунок всегда одно условие.
REPLICATE_LAYOUT = "auto"

# =========================
# НАСТРОЙКИ ФИЛЬТРА КОЛОНИЙ
# =========================

# Минимальный размер колонии в пикселях. Начните с 50.
# Если считается много мусора, увеличьте: 60, 80, 100.
# Если пропадают настоящие маленькие колонии, уменьшите: 40, 32, 25.
MIN_COLONY_AREA = 50

# Минимальная средняя локальная темнота/контраст.
# Если проходят бледные пятна фона, увеличьте: 0.28, 0.32.
# Если пропадают слабые, но настоящие колонии, уменьшите: 0.20, 0.18.
MIN_MEAN_BLACKHAT = 0.24

# Верхняя граница обычной одиночной колонии.
# Более крупные хорошо окрашенные объекты считаются как large_clump, то есть как одна слипшаяся колония.
MAX_SINGLE_COLONY_AREA = 900

# Строить CDI и heatmap. Для этого нужен контроль 0 Gy + концентрация 0.
RUN_CDI = True

# Показывать после анализа несколько overlay-картинок прямо в ноутбуке.
SHOW_EXAMPLE_OVERLAYS = True

In [ ]:
# Установка кода и зависимостей
import os
import shutil
import subprocess
import sys
from pathlib import Path

repo_dir = Path("Clonogenic")
if not repo_dir.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "requirements.txt")], check=True)

photo_dir = repo_dir / PHOTO_DIR
output_dir = repo_dir / OUTPUT_DIR
photo_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print("Готово. Репозиторий:", repo_dir.resolve())
print("Папка для фото:", photo_dir.resolve())
print("Папка результатов:", output_dir.resolve())

## Загрузка фотографий

Подготовьте ZIP-архив с фотографиями. Внутри архива могут быть `.jpg`, `.jpeg`, `.png`, `.tif`, `.tiff`.

После запуска следующей ячейки выберите ZIP-файл на компьютере.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Файл не загружен")

# Очистить старые фото, чтобы случайно не смешать разные эксперименты.
for old_file in photo_dir.glob("**/*"):
    if old_file.is_file():
        old_file.unlink()

for filename in uploaded:
    archive_path = Path(filename)
    if archive_path.suffix.lower() != ".zip":
        raise RuntimeError("Загрузите ZIP-архив, а не отдельный файл: " + filename)
    with zipfile.ZipFile(archive_path, "r") as archive:
        archive.extractall(photo_dir)

image_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
images = sorted(path for path in photo_dir.rglob("*") if path.suffix.lower() in image_extensions)
print("Найдено изображений:", len(images))
for path in images[:20]:
    print("-", path.relative_to(photo_dir))

In [ ]:
# Запуск подсчета колоний
image_patterns = [str(photo_dir / "**" / "*.jpg"), str(photo_dir / "**" / "*.jpeg"), str(photo_dir / "**" / "*.png"), str(photo_dir / "**" / "*.tif"), str(photo_dir / "**" / "*.tiff")]

cmd = [
    sys.executable,
    str(repo_dir / "scripts" / "analyze_clonogenic_photo.py"),
    *image_patterns,
    "--output-dir", str(output_dir),
    "--min-area", str(MIN_COLONY_AREA),
    "--max-area", str(MAX_SINGLE_COLONY_AREA),
    "--min-mean-blackhat", str(MIN_MEAN_BLACKHAT),
    "--replicate-layout", REPLICATE_LAYOUT,
]
print("Команда:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("Анализ завершен:", output_dir)

In [ ]:
# Просмотр таблицы по лункам
import pandas as pd

well_counts_path = output_dir / "batch_well_counts.csv"
well_counts = pd.read_csv(well_counts_path)
display(well_counts)

print("Файл:", well_counts_path)

In [ ]:
# Показать несколько overlay-картинок с номерами колоний
from IPython.display import Image, display

if SHOW_EXAMPLE_OVERLAYS:
    overlays = sorted(output_dir.glob("*/plate_overlay.png"))[:3]
    for overlay in overlays:
        print(overlay.parent.name)
        display(Image(filename=str(overlay), width=900))

In [ ]:
# CDI и heatmap
if RUN_CDI:
    cdi_prefix = output_dir.parent / "photo_cdi"
    subprocess.run([
        sys.executable,
        str(repo_dir / "scripts" / "build_cdi_table.py"),
        "--input", f"{CELL_LINE_NAME}={output_dir}",
        "--material", MATERIAL_NAME,
        "--output-prefix", str(cdi_prefix),
        "--skip-verification",
    ], check=True)
    heatmap_path = output_dir.parent / "photo_cdi_heatmap.png"
    subprocess.run([
        sys.executable,
        str(repo_dir / "scripts" / "heatmap.py"),
        "--input", str(cdi_prefix.with_name(cdi_prefix.name + "_long.csv")),
        "--output", str(heatmap_path),
        "--material", MATERIAL_NAME,
        "--value-field", "cdi_100",
    ], check=True)
    display(pd.read_csv(cdi_prefix.with_name(cdi_prefix.name + "_long.csv")))
    display(Image(filename=str(heatmap_path), width=800))
else:
    print("RUN_CDI = False, CDI не строится")

In [ ]:
# Скачать все результаты ZIP-архивом
zip_base = repo_dir / "photo_results"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=output_dir.parent, base_dir=output_dir.name)
print("ZIP:", zip_path)
files.download(zip_path)